# Summarize a hyperparameter sweep

A sweep trains each configuration with several random seeds and writes one line per run to a
log, as `config | seed | accuracy`. Some lines are broken, and one seed was run twice because the
first attempt was interrupted. Here you turn the raw log into a ranking of configurations by their
mean accuracy over seeds, in plain Python.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import math

sweep_log = [
    "lr0.1-bs32 | 1 | 0.842",
    "lr0.1-bs32 | 2 | 0.851",
    "lr0.01-bs32 | 1 | 0.867",
    "lr0.01-bs32 | 2 | n/a",
    "lr0.01-bs32 | 3 | 0.873",
    "lr0.1-bs64 | 1 | 0.858",
    "lr0.01-bs32 | 1 | 0.849",
    "lr0.1-bs64 | 2 | 1.858",
    "lr0.1-bs64 | 3 | 0.846",
    "lr0.001-bs32 | 1 | 0.890",
    "lr0.1-bs32 | 3",
    "lr0.1-bs64 | two | 0.850",
]
for line in sweep_log:
    print(line)

## Part 1 (coding task): Parse one line

Write `parse_line(line)`. For a good line it returns the tuple `(config, seed, acc)`: the
configuration name as a string with no spaces around it, the seed as an `int` and the accuracy as a
`float`. `" lr0.1 ".strip()` removes the spaces at both ends of a string.

A line is broken, and `parse_line` returns `None`, when it does not have exactly three fields, when
the seed is not a whole number, when the accuracy is not a number, or when the accuracy lies outside
$[0, 1]$. Broken lines must never raise an error. Watch for `"nan"`: `float("nan")` parses without
error and gives the special value "not a number", so only the range test can catch it.

In [ ]:
def parse_line(line):
    """(config, seed, acc) for a good line, None for a broken one."""
    # TODO
    raise NotImplementedError

In [ ]:
def _try_parse(text):
    try:
        return parse_line(text)
    except NotImplementedError:
        raise
    except Exception as err:
        raise AssertionError(f"parse_line raised {type(err).__name__} on {text!r}: a broken line should give None") from None

good = _try_parse("lr0.1-bs32 | 2 | 0.851")
assert good is not None, "a good line gave None"
assert good[0] == "lr0.1-bs32", f"config {good[0]!r}: strip the spaces around the name"
assert type(good[1]) is int, f"the seed should be an int, got {type(good[1]).__name__}"
assert type(good[2]) is float, f"the accuracy should be a float, got {type(good[2]).__name__}"
assert good == ("lr0.1-bs32", 2, 0.851), f"expected ('lr0.1-bs32', 2, 0.851), got {good}"
assert _try_parse("perfect | 7 | 1.0") == ("perfect", 7, 1.0), "an accuracy of exactly 1 is allowed"
for bad, why in [
    ("lr0.01-bs32 | 2 | n/a", "the accuracy is not a number"),
    ("lr0.1-bs32 | 3", "a field is missing"),
    ("lr0.1-bs32 | 3 | 0.8 | late", "there is an extra field"),
    ("lr0.1-bs64 | two | 0.850", "the seed is not a whole number"),
    ("lr0.1-bs64 | 2.5 | 0.850", "the seed is not a whole number"),
    ("lr0.1-bs64 | 2 | 1.858", "an accuracy above 1 is impossible"),
    ("lr0.1-bs64 | 2 | -0.2", "a negative accuracy is impossible"),
    ("lr0.1-bs64 | 2 | nan", "nan is not an accuracy, and it fails every comparison"),
]:
    got = _try_parse(bad)
    assert got is None, f"{bad!r} should give None because {why}, got {got}"
print("Part 1 passed")

## Part 2 (coding task): Group the runs by configuration

Write `summarize(lines)`, which returns a dictionary that maps each configuration to a tuple
`(mean_acc, n_seeds)`: the mean accuracy over its seeds and the number of seeds. Skip broken lines.
When the same configuration and seed appear twice, the later line is a rerun and replaces the
earlier one, so each seed counts once.

In [ ]:
def summarize(lines):
    """config -> (mean accuracy over its seeds, number of seeds)."""
    # TODO
    raise NotImplementedError

In [ ]:
summary = summarize(sweep_log)
print(summary)
assert isinstance(summary, dict), "summarize should return a dict"
assert set(summary) == {"lr0.1-bs32", "lr0.01-bs32", "lr0.1-bs64", "lr0.001-bs32"}, f"configs {sorted(summary)}"
mean, n = summary["lr0.01-bs32"]
assert n != 3, "lr0.01-bs32 has seed 1 twice: the rerun replaces the first run, so it has 2 seeds"
assert n == 2, f"lr0.01-bs32 should have 2 seeds, got {n}"
assert math.isclose(mean, 0.861), f"lr0.01-bs32 mean {mean}: seeds 1 (rerun, 0.849) and 3 (0.873) average 0.861"
mean, n = summary["lr0.1-bs64"]
assert n == 2 and math.isclose(mean, 0.852), f"lr0.1-bs64 gave {(mean, n)}: its seed 2 and 'two' lines are broken"
assert summary["lr0.001-bs32"] == (0.89, 1), f"lr0.001-bs32 gave {summary['lr0.001-bs32']}"
assert math.isclose(summary["lr0.1-bs32"][0], 0.8465) and summary["lr0.1-bs32"][1] == 2, f"lr0.1-bs32 gave {summary['lr0.1-bs32']}"
assert summarize([]) == {}, "an empty log gives an empty summary"
rerun = summarize(["a | 1 | 0.5", "a | 1 | 0.7", "a | 2 | 0.9"])
assert rerun["a"][1] == 2 and math.isclose(rerun["a"][0], 0.8), f"{rerun}: a rerun replaces the earlier run of the same seed"
print("Part 2 passed")

## Part 3 (coding task): Rank the configurations

Write `ranking(summary, min_seeds)`, which returns the configuration names that have at least
`min_seeds` seeds, best first: by mean accuracy from highest to lowest, then, for equal means, by more
seeds first, then by name in alphabetical order. A configuration that only one lucky seed has
measured should not win a ranking that asks for two.

In [ ]:
def ranking(summary, min_seeds):
    """Names with at least min_seeds seeds, best first."""
    # TODO
    raise NotImplementedError

In [ ]:
summary = summarize(sweep_log)
top = ranking(summary, 2)
print(top)
assert "lr0.001-bs32" not in top, "lr0.001-bs32 has one seed, so min_seeds=2 leaves it out"
assert top == ["lr0.01-bs32", "lr0.1-bs64", "lr0.1-bs32"], f"expected ['lr0.01-bs32', 'lr0.1-bs64', 'lr0.1-bs32'], got {top}"
assert ranking(summary, 1)[0] == "lr0.001-bs32", "with min_seeds=1 the single 0.890 run comes first"
assert ranking(summary, 3) == [], "no configuration has three seeds"
ties = {"beta": (0.9, 2), "alpha": (0.9, 2), "gamma": (0.9, 3), "delta": (0.8, 5)}
got = ranking(ties, 1)
assert got[0] == "gamma", f"{got}: among equal means, more seeds come first"
assert got != ["gamma", "beta", "alpha", "delta"], "equal means and seeds go alphabetically, but these names are in reverse order: reverse=True flips the name order too"
assert got == ["gamma", "alpha", "beta", "delta"], f"expected ['gamma', 'alpha', 'beta', 'delta'], got {got}"
print("Part 3 passed")

## Solution

### Part 1: Parse one line

In [ ]:
def parse_line(line):
    """(config, seed, acc) for a good line, None for a broken one."""
    fields = line.split("|")
    if len(fields) != 3:
        return None
    try:
        seed = int(fields[1])
        acc = float(fields[2])
    except ValueError:
        return None
    if not 0 <= acc <= 1:
        return None
    return fields[0].strip(), seed, acc

**Why it works.** Splitting at `|` and counting the pieces catches missing and extra fields before anything is
converted. `int` and `float` both ignore spaces at the ends of the text, and both raise
`ValueError` on text they cannot read, so one `except ValueError` covers `"n/a"`, `"two"` and
`"2.5"`. The range test `0 <= acc <= 1` also rejects `nan`, because every comparison with `nan` is
`False`. Only the name needs `strip()`, since nothing else would remove its spaces.

### Part 2: Group the runs by configuration

In [ ]:
def summarize(lines):
    """config -> (mean accuracy over its seeds, number of seeds)."""
    by_config = {}                     # config -> {seed: acc}
    for line in lines:
        parsed = parse_line(line)
        if parsed is None:
            continue
        config, seed, acc = parsed
        by_config.setdefault(config, {})[seed] = acc
    return {
        config: (sum(accs.values()) / len(accs), len(accs))
        for config, accs in by_config.items()
    }

**Why it works.** A dictionary of dictionaries, configuration to seed to accuracy, does the grouping, and storing
under a seed that is already there replaces its value, which is exactly the rerun rule. Counting
the seeds afterwards with `len` then counts each seed once. `setdefault(config, {})` returns the
inner dictionary, creating it on first use, so one line both groups and stores. Averaging per line
as you go would count the interrupted run of `lr0.01-bs32` seed 1 as a third seed.

### Part 3: Rank the configurations

In [ ]:
def ranking(summary, min_seeds):
    """Names with at least min_seeds seeds, best first."""
    kept = [(name, mean, n) for name, (mean, n) in summary.items() if n >= min_seeds]
    kept = sorted(kept, key=lambda r: (-r[1], -r[2], r[0]))
    return [name for name, mean, n in kept]

**Why it works.** The key is a tuple, compared entry by entry. A minus sign makes the higher mean and the larger seed
count come first, and the name stays as it is, so ties go alphabetically. `reverse=True` with the
key `(mean, n, name)` would get the first two entries right and the names backwards, which is the
mistake the tie check looks for. On the sweep, `lr0.01-bs32` ranks first at 0.861 once the single
0.890 run of `lr0.001-bs32` is filtered out.